# 06b - A First K-Means Example

We want to know whether patterns of customer requests might help a service team plan its support. Each row below represents one customer. We record how many requests that customer made last term and the average number of minutes each request took.

The data are simulated for this demonstration. We have not assigned customers to types or built clusters into the simulation. Our question is what K-means proposes when we ask for two or three groups, and what we would still need to learn before using either grouping.

## References

- [ISLP, Chapter 12 lab](https://islp.readthedocs.io/en/latest/labs/Ch12-unsup-lab.html): a first Python example of K-means and its fitted labels.
- [Hands-On Machine Learning, Chapter 9 notebook](https://github.com/ageron/handson-ml3/blob/main/09_unsupervised_learning.ipynb): a visual walkthrough of successive K-means updates.
- [Scikit-learn clustering guide](https://scikit-learn.org/stable/modules/clustering.html#k-means): the algorithm, its distance criterion, and its limits.

## In the Notebook

Before we fit the customer data, watch K-means work on ten small points. The two starting centers are selected randomly from those points. In each update, every point is assigned to its nearest current center, then each center moves to the mean of its assigned points. The dot colors show the assignments used for that move, and the arrows show where the centers moved. The final panel shows the settled result.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

toy_points = np.array(
    [[0, 0], [0, 1], [1, 0], [1, 1], [3, 3], [4, 4], [5, 3], [5, 4], [7, 1], [8, 1]],
    dtype=float,
)
toy_rng = np.random.default_rng(0)
toy_centers = toy_points[toy_rng.choice(len(toy_points), size=2, replace=False)].copy()
toy_frames = [(toy_centers.copy(), None, None)]

for _ in range(20):
    squared_distances = ((toy_points[:, None, :] - toy_centers[None, :, :]) ** 2).sum(axis=2)
    toy_labels = squared_distances.argmin(axis=1)
    moved_centers = np.array([toy_points[toy_labels == group].mean(axis=0) for group in range(2)])
    if np.allclose(moved_centers, toy_centers):
        break
    toy_frames.append((moved_centers.copy(), toy_labels.copy(), toy_centers.copy()))
    toy_centers = moved_centers

shown_frames = [toy_frames[0], toy_frames[1], toy_frames[2], toy_frames[-1]]
panel_titles = ["Random start", "After update 1", "After update 2", "Settled"]
group_colors = ["#4477AA", "#EE6677"]

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)
for ax, title, (centers, labels, previous) in zip(
    axes.flat, panel_titles, shown_frames, strict=True
):
    if labels is None:
        ax.scatter(toy_points[:, 0], toy_points[:, 1], c="0.65", s=55, edgecolors="black")
    else:
        for group, color in enumerate(group_colors):
            points = toy_points[labels == group]
            ax.scatter(points[:, 0], points[:, 1], c=color, s=55, edgecolors="black")
            ax.annotate(
                "", xy=centers[group], xytext=previous[group],
                arrowprops={"arrowstyle": "->", "color": color, "lw": 2},
            )
    ax.scatter(centers[:, 0], centers[:, 1], c=group_colors, marker="X", s=180, edgecolors="black")
    ax.set(xlim=(-0.7, 8.7), ylim=(-0.7, 4.7), title=title, aspect="equal")
    ax.grid(alpha=0.15)
for ax in axes[1]:
    ax.set_xlabel("Feature 1")
for ax in axes[:, 0]:
    ax.set_ylabel("Feature 2")
fig.suptitle("Assignments and center updates on the same ten points")
fig.tight_layout()
plt.show()

Both centers start near $(5, 3)$ to $(5, 4)$. After the first update, one moves toward the left-hand points; after the second, the centers separate further. The settled centers summarize the two final assignments. The starting locations shape the path the algorithm takes.

## Customer Support Example

The next cell creates 72 customer records. Every customer has at least one request, so an average time per request is defined. The two characteristics vary across broad ranges, but the simulation contains no predefined customer groups. Inspect the scatterplot before fitting a model: do you see clear gaps that establish a number of groups?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(7130)
customers = pd.DataFrame(
    {
        "requests": rng.integers(1, 13, size=72),
        "avg_minutes": rng.integers(8, 65, size=72),
    }
)
customers.index = pd.Index(range(1, len(customers) + 1), name="customer")

display(customers.head())

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(customers["requests"], customers["avg_minutes"], alpha=0.75)
ax.set(
    xlabel="Requests last term",
    ylabel="Average minutes per request",
    title="Simulated customers",
)
ax.grid(alpha=0.2)
plt.show()

These 72 points describe a spread of customer behavior. The plot does not supply known group labels or a reason to treat any proposed group differently.

## Fit K-Means With Two and Three Groups

K-means uses distances between rows. The minute values span a much wider numerical range than the request counts, so we standardize both columns before fitting. `StandardScaler` subtracts each column's mean and divides by its standard deviation. We then fit one K-means model with `n_clusters=2` and another with `n_clusters=3`. The fixed `random_state` makes this demonstration reproducible; `n_init=20` tries multiple randomized starts for each value of K.

The plots return to the original units so we can read the customers' behavior. Each colored area shows which group the fitted model would assign to a customer at that location; the black dots are the observed customers, and each black X marks a group center. To draw the areas, we apply the same scaling to a grid of possible values before asking each model for its assignments. Group numbers and colors are local to each panel; group 0 on the left is not automatically group 0 on the right.

In [ ]:
features = customers[["requests", "avg_minutes"]]
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

models = {}
assignments = {}
for k in (2, 3):
    model = KMeans(n_clusters=k, random_state=7130, n_init=20)
    assignments[k] = model.fit_predict(scaled_features)
    models[k] = model

x_values = np.linspace(0.5, 12.5, 240)
y_values = np.linspace(5, 67, 240)
grid_x, grid_y = np.meshgrid(x_values, y_values)
grid = pd.DataFrame(
    {"requests": grid_x.ravel(), "avg_minutes": grid_y.ravel()}
)
scaled_grid = scaler.transform(grid)
region_colors = ListedColormap(["#F3C7E3", "#FFF0AD", "#B9E6D3"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)
for ax, k in zip(axes, (2, 3), strict=True):
    centers = scaler.inverse_transform(models[k].cluster_centers_)
    regions = models[k].predict(scaled_grid).reshape(grid_x.shape)
    ax.contourf(grid_x, grid_y, regions, levels=np.arange(-0.5, k + 0.5, 1), cmap=region_colors)
    ax.contour(grid_x, grid_y, regions, levels=np.arange(0.5, k, 1), colors="black", linewidths=0.8)
    ax.scatter(features["requests"], features["avg_minutes"], s=18, c="black", alpha=0.7)
    ax.scatter(centers[:, 0], centers[:, 1], marker="X", s=180, c="black", label="Group center")
    for group_id, (center_x, center_y) in enumerate(centers):
        ax.annotate(str(group_id), (center_x, center_y), xytext=(8, 8), textcoords="offset points")
    ax.set(xlabel="Requests last term", title=f"K = {k}")
    ax.legend(loc="upper right")
axes[0].set_ylabel("Average minutes per request")
fig.suptitle("Two proposed ways to group the same customers")
fig.tight_layout()
plt.show()

Both settings assign every customer to a group. The colored regions change when we ask for a different number of groups. Neither picture establishes that the service team needs two or three kinds of support.

## Compare the Group Profiles

The table below reports the number of customers in each fitted group and their mean values in the original units. Read each row as a description of the customers assigned to that group, not as a label they had before the analysis.

In [ ]:
profiles = []
for k in (2, 3):
    grouped = customers.assign(group=assignments[k]).groupby("group")
    summary = grouped.agg(
        customers=("requests", "size"),
        mean_requests=("requests", "mean"),
        mean_minutes=("avg_minutes", "mean"),
    )
    summary.insert(0, "K", k)
    profiles.append(summary.reset_index())

profile_table = pd.concat(profiles, ignore_index=True)
display(profile_table.round({"mean_requests": 1, "mean_minutes": 1}))

With K = 2, the groups differ mainly in request frequency: one averages about 9.2 requests and the other about 3.4, while their average request times are similar. With K = 3, one group averages about 53 minutes per request, compared with about 27 to 28 minutes for the other two groups. Those distinctions describe this simulated table. They do not tell us whether the service team should change its support.

## What Would Earn a Third Group?

Compare the two profile tables. What possible service decision could make the longer-request group worth separate attention? What information beyond these two features would you want before acting on that distinction?

##### Answer

The longer-request group might prompt the team to examine whether those customers need more specialized help or longer appointments. The table alone does not show that either response would work. We would want to inspect actual requests, ask the service team whether the distinction matches its experience, and check whether a similar pattern appears in another period. Because this dataset is simulated, it provides no independent evidence about real customers or outcomes.

## When Another Group Is Just a Split

In this final simulation, points form two visible clouds, with one cloud more spread out than the other. Fit K-means with K = 2 and K = 3. Colors identify assignments separately in each panel. Before looking at how the points were generated, ask whether the three-group result is evidence of a third underlying population.

In [ ]:
from sklearn.datasets import make_blobs

two_clouds, source_population = make_blobs(
    n_samples=[50, 50],
    centers=[(-2, 0), (2, 0)],
    cluster_std=[0.35, 0.8],
    random_state=7130,
)
cloud_models = {
    k: KMeans(n_clusters=k, n_init=20, random_state=7130).fit(two_clouds)
    for k in (2, 3)
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
for ax, k in zip(axes, (2, 3), strict=True):
    model = cloud_models[k]
    ax.scatter(two_clouds[:, 0], two_clouds[:, 1], c=model.labels_, cmap="Set2", s=24)
    ax.scatter(
        model.cluster_centers_[:, 0], model.cluster_centers_[:, 1],
        c="black", marker="X", s=170,
    )
    ax.set(xlabel="Feature 1", title=f"K = {k}")
    ax.grid(alpha=0.15)
axes[0].set_ylabel("Feature 2")
fig.suptitle("K-means partitions the same two clouds in two ways")
fig.tight_layout()
plt.show()

The simulation generated 50 points from each of **two** populations. It used different spreads, but it did not generate a third population. The table checks where the K = 3 assignments came from.

In [ ]:
origin_check = pd.crosstab(
    pd.Series(source_population, name="Generating population"),
    pd.Series(cloud_models[3].labels_, name="K = 3 group"),
)
display(origin_check)

K = 3 splits the wider generated population into two groups. The groups differ among these observed points, but this result does not reveal a third generating population. In real data we would not know the source populations in advance, which makes the choice of K an evidence question rather than something the algorithm settles for us.